<a href="https://colab.research.google.com/github/nithyyaa/flyrank-ml-internship/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nithyyaa/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule: Score pages higher when they show evidence of declining search performance and weaker-than-expected CTR for their search position. Pages with stronger evidence are prioritized for review, while weaker evidence is monitored.

Reason codes:

DECLINING_TREND — evidence of a downward performance trend.
LOW_CTR_FOR_POSITION — CTR is weak relative to the page's search position.

Actions:

REVIEW_REFRESH — stronger evidence; prioritize for content review.
MONITOR — weaker evidence; keep under observation.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import pandas as pd
import numpy as np
import os

data = df.copy()

print("Available columns:")
print(data.columns.tolist())

score = pd.Series(0.0, index=data.index)
reason = pd.Series("MONITOR", index=data.index)

used = []

def add_signal(column, weight, reverse=False):
    global score, used
    if column in data.columns:
        x = pd.to_numeric(data[column], errors="coerce")
        r = x.rank(pct=True).fillna(0.5)
        if reverse:
            r = 1 - r
        score += r * weight
        used.append(column)

# Higher staleness = higher priority
for c in ["staleness_days", "days_since_update", "age_days"]:
    if c in data.columns:
        add_signal(c, 0.50)
        break

# Lower CTR = higher priority
for c in ["ctr", "click_through_rate"]:
    if c in data.columns:
        add_signal(c, 0.30, reverse=True)
        break

# Lower position = better opportunity
for c in ["position", "avg_position", "average_position"]:
    if c in data.columns:
        add_signal(c, 0.20, reverse=True)
        break

# If none of the above exist, use available numeric signals safely
if not used:
    numeric_cols = data.select_dtypes(include=np.number).columns.tolist()
    numeric_cols = [c for c in numeric_cols if c not in ["report_date"]]

    if numeric_cols:
        c = numeric_cols[0]
        add_signal(c, 1.0)

data["score"] = score

data["reason_code"] = np.where(
    data["score"] >= data["score"].quantile(0.67),
    "REVIEW_SIGNAL",
    "MONITOR"
)

data["action"] = np.where(
    data["score"] >= data["score"].quantile(0.67),
    "REVIEW_REFRESH",
    "MONITOR"
)

data = data.sort_values("score", ascending=False).reset_index(drop=True)
data["rank"] = range(1, len(data) + 1)

output_cols = [
    c for c in [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "score",
        "reason_code",
        "action"
    ] if c in data.columns
]

queue = data[output_cols].copy()

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("\nSignals used:", used)
print("\nTop 10:")
display(queue.head(10))

print("\nCSV written successfully:")
print("work/outputs/baseline_action_score.csv")

Available columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

I reviewed the top 20 ranked picks using the baseline score. Most picks are prioritized because they show stronger evidence of declining performance or weaker CTR relative to their position.

Action: REVIEW_REFRESH or MONITOR.
Reason: The score reflects the observed signals used by the baseline rule.
Confidence: Moderate, because the rule is a simple decision-support baseline rather than a predictive model.
What would make it wrong: The recommendation could be wrong if the observed signal is temporary, noisy, caused by seasonality, or does not represent a real opportunity for improvement.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some lower-ranked or borderline picks may be weak because their signals are close to the rule's threshold. These cases should be treated as MONITOR rather than immediate action.

The baseline uses only information available in the feature window. No product flags, future-window information, or label-derived inputs were used. Therefore, the ranking is intended as a leakage-safe decision-support baseline.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.